# Day 032 Project: Secure Config Module

## What You're Building

A `SecureConfig` instance loaded from a `.env` text string that:
1. Parses the .env format with `parse_dotenv`
2. Validates required keys at startup with `validate`
3. Exposes values safely with `get` and `require`
4. Produces a log-safe representation with `masked_dict`

## Project Requirements

1. Define an `ENV_TEXT` string in .env format with at least 4 keys including
   at least one secret (e.g., `API_KEY`, `TOKEN`, or `DB_PASSWORD`)
2. Parse it with `parse_dotenv` and load into a `SecureConfig` instance
3. Store the instance as `cfg`
4. Run `validate` against at least 3 required keys
5. Print `cfg.masked_dict(secret_keys)` to verify safe logging
6. Verify with `_run_project_checks()`

## Provided: All Helper Functions + SecureConfig

In [ ]:
def parse_dotenv(text: str) -> dict:
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        key   = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        result[key] = value
    return result


def mask_secret(value, show_chars: int = 4) -> str:
    s = str(value)
    if len(s) <= show_chars:
        return "***"
    return s[:show_chars] + "***" 


def validate_config(config: dict, required_keys: list) -> list:
    return [k for k in required_keys if config.get(k) is None]


def safe_log_config(config: dict, secret_keys: list) -> dict:
    return {
        k: mask_secret(str(v)) if k in secret_keys else v
        for k, v in config.items()
    }


class SecureConfig:
    def __init__(self, defaults: dict | None = None):
        self._config: dict = dict(defaults or {})

    def load_dict(self, mapping: dict) -> "SecureConfig":
        self._config.update(mapping)
        return self

    def get(self, key: str, default=None):
        return self._config.get(key, default)

    def require(self, key: str) -> str:
        val = self._config.get(key)
        if val is None:
            raise KeyError(f"Required config key not found: '{key}'")
        return str(val)

    def validate(self, required_keys: list) -> list:
        return validate_config(self._config, required_keys)

    def masked_dict(self, secret_keys: list) -> dict:
        return safe_log_config(self._config, secret_keys)

## Your Config Setup

Define your .env text, required keys, and secret keys below.

In [ ]:
# Define your .env content
ENV_TEXT = """
# Application configuration
API_KEY=sk-demo-key-12345678
DB_HOST=localhost
DB_PORT=5432
DB_NAME=myapp
MODEL_NAME=llama3.2
DEBUG=false
"""

# Keys that are secrets (will be masked in logs)
SECRET_KEYS = ['API_KEY']

# Keys that must be present at startup
REQUIRED_KEYS = ['API_KEY', 'DB_HOST', 'DB_NAME', 'MODEL_NAME']

# Build config
cfg = SecureConfig(defaults={'DEBUG': 'false', 'MODEL_NAME': 'llama3.2'})
cfg.load_dict(parse_dotenv(ENV_TEXT))

# Validate
missing = cfg.validate(REQUIRED_KEYS)
if missing:
    raise EnvironmentError(f'Missing required config: {missing}')
print('All required keys present')

# Safe logging
safe = cfg.masked_dict(SECRET_KEYS)
print('\nConfig (safe for logging):')
for k, v in safe.items():
    print(f'  {k} = {v}')

# Access values
print(f'\nDB host:   {cfg.require("DB_HOST")}')
print(f'Model:     {cfg.require("MODEL_NAME")}')
print(f'API key:   {mask_secret(cfg.require("API_KEY"))}')

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: cfg is a SecureConfig instance
    try:
        assert 'cfg' in globals()
        assert isinstance(cfg, SecureConfig), \
            f'cfg should be SecureConfig, got {type(cfg)}'
        passed += 1; print('\u2705 Check 1: cfg is a SecureConfig instance')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: cfg has at least 4 loaded keys
    try:
        assert hasattr(cfg, '_config')
        assert len(cfg._config) >= 4, \
            f'expected >=4 config keys, got {len(cfg._config)}'
        passed += 1; print(f'\u2705 Check 2: {len(cfg._config)} keys loaded')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: validate returns [] (all required keys present)
    try:
        missing = cfg.validate(REQUIRED_KEYS)
        assert missing == [], \
            f'missing required keys: {missing}'
        passed += 1; print('\u2705 Check 3: all required keys present')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: masked_dict masks secret keys
    try:
        safe = cfg.masked_dict(SECRET_KEYS)
        for sk in SECRET_KEYS:
            assert '***' in str(safe.get(sk, '')), \
                f"secret key '{sk}' not masked: {safe.get(sk)!r}"
        passed += 1; print(f'\u2705 Check 4: secret keys masked in masked_dict')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: require raises on a non-existent key
    try:
        raised = False
        try:
            cfg.require('DEFINITELY_NOT_SET_XYZ')
        except KeyError:
            raised = True
        assert raised, 'require should raise KeyError for missing key'
        passed += 1; print('\u2705 Check 5: require raises KeyError for missing key')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Extend SecureConfig with a `from_env(path='.env')` classmethod that calls `load_dotenv()` (python-dotenv) and then `load_dict(os.environ)` in one step
- Add a `save_template(path)` method that writes a .env.example file with keys but empty values — useful for documenting required config in a repository
- Add a `require_all(keys)` method that calls require for every key and collects all KeyErrors into one EnvironmentError — better UX than one error per missing key
- Add type coercion to get: `get_int(key, default=0) -> int`, `get_bool(key, default=False) -> bool` using 'true'/'1'/'yes' for True
- Integrate SecureConfig into a Day 31 batch processor: load the process_fn URL from config with require() instead of hardcoding it